In [3]:
import ee

ee.Authenticate()


Successfully saved authorization token.


In [1]:
# 2. Create a Dask cluster

from dask.distributed import Client, LocalCluster

cluster = LocalCluster(
    n_workers=4,                  # number of Python processes
    threads_per_worker=1,         # single-threaded workers
    memory_limit="3GB",           # per worker; can be "auto"
    scheduler_port=0,             # 0 = choose a free port
    dashboard_address=":8787",    # open the dashboard
    processes=True,               # True = separate processes (default)
)
client = Client(cluster)

In [4]:
# 3. Initialize Dask workers with Google Earth Engine
import ee
from distributed import WorkerPlugin

class EEPlugin(WorkerPlugin):
    def __init__(self):
        pass
    def setup(self, worker):
        self.worker = worker
        try:
            # Assume credentials already exist at default location
            ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')
        except ee.EEException as e:
            raise RuntimeError("Earth Engine initialization failed. "
                            "Run ee.Authenticate() once on the client before starting the cluster.")

ee_plugin = EEPlugin()
client.register_plugin(ee_plugin)

# 4. Run some Earth Engine code! This will do a basic grab of some Landsat data, with some standard filtering. 
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

US_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/US")
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-01-01', '2018-12-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])

In [5]:
import xarray as xr

ds = xr.open_dataset(ic, engine='ee', geometry=US_Boundaries.geometry(), crs='EPSG:5070', scale=1000, chunks='auto')

# 6. Compute NDVI using the xarray
import numpy as np

def add_ndvi(ds: xr.Dataset) -> xr.Dataset:
    # Bands: B5 = NIR, B4 = Red
    nir = ds["SR_B5"].astype("float32")
    red = ds["SR_B4"].astype("float32")

    # Compute NDVI, guard against divide-by-zero
    denom = nir + red
    ndvi = xr.where(denom != 0, (nir - red) / denom, np.nan).astype("float32")

    # Add attributes for clarity
    ndvi = ndvi.assign_attrs(
        {
            "long_name": "Normalized Difference Vegetation Index",
            "standard_name": "NDVI",
            "description": "NDVI = (SR_B5 - SR_B4) / (SR_B5 + SR_B4)",
            "units": "1",
            "source_bands": "SR_B5 (NIR), SR_B4 (Red)",
        }
    )

    # Mutate dataset directly
    return xr.Dataset({"ndvi": ndvi})

# 7. Run the computation, which should replicate the warning.
results = xr.map_blocks(func=add_ndvi, obj=ds)

In [7]:
graph = results.__dask_graph__()

In [10]:
graph

HighLevelGraph with 6 layers.
<dask.highlevelgraph.HighLevelGraph object at 0x1b1968e7a30>
 0. original-open_dataset-SR_B5-8c483b89182026ab89cd54a451d58219
 1. open_dataset-SR_B5-8c483b89182026ab89cd54a451d58219
 2. original-open_dataset-SR_B4-8c483b89182026ab89cd54a451d58219
 3. open_dataset-SR_B4-8c483b89182026ab89cd54a451d58219
 4. add_ndvi-f1c227c4ea0639a577ca92bb0a5e71f9
 5. ndvi-add_ndvi-f1c227c4ea0639a577ca92bb0a5e71f9

In [8]:
n_tasks = len(graph.keys())
print(n_tasks)

2143646


In [11]:
import numpy as np

def chunk_bounds(chunks):
    # chunks is a tuple of chunk sizes, e.g. ds.chunks['time']
    edges = np.cumsum((0,)+tuple(chunks))
    return list(zip(edges[:-1], edges[1:]))

time_chunks = results.chunks["time"]          # tuple of sizes per time-chunk
bounds = chunk_bounds(time_chunks)

In [ ]:
import numpy as np

def chunk_bounds(chunks):
    # chunks is a tuple of chunk sizes, e.g. ds.chunks['time']
    edges = np.cumsum((0,)+tuple(chunks))
    return list(zip(edges[:-1], edges[1:]))

time_chunks = results.chunks["time"]          # tuple of sizes per time-chunk
bounds = chunk_bounds(time_chunks)
print(bounds)
# adaptively pack N time-chunks so each batch stays under ~100k tasks
TASK_LIMIT = 100_000
i = 0
while i < len(bounds):
    j = i + 1
    # grow the window until we would exceed the task limit
    while j <= len(bounds):
        t0, _ = bounds[i]
        _, t1 = bounds[j-1]
        batch = results['ndvi'].isel(time=slice(t0, t1))  # 🔹 target a single variable
        n = len(batch.__dask_graph__().keys())
        print(n)
        if n > TASK_LIMIT:
            j -= 1
            break
    j += 1
    if j == i:  # even a single time-chunk exceeds the limit; just run it
        j = i + 1
        t0, _ = bounds[i]; _, t1 = bounds[j-1]
        batch = results.isel(time=slice(t0, t1))
    # do the work for this batch (write/export/compute)
    #_ = batch.compute()   # or persist + export + cancel
    # free refs before next batch
    del batch
    i = j


In [12]:
print(bounds)

[(0, 48), (48, 96), (96, 144), (144, 192), (192, 240), (240, 288), (288, 336), (336, 384), (384, 432), (432, 480), (480, 528), (528, 576), (576, 624), (624, 672), (672, 720), (720, 768), (768, 816), (816, 864), (864, 912), (912, 960), (960, 1008), (1008, 1056), (1056, 1104), (1104, 1152), (1152, 1200), (1200, 1248), (1248, 1296), (1296, 1344), (1344, 1392), (1392, 1440), (1440, 1488), (1488, 1536), (1536, 1584), (1584, 1632), (1632, 1680), (1680, 1728), (1728, 1776), (1776, 1824), (1824, 1872), (1872, 1920), (1920, 1968), (1968, 2016), (2016, 2064), (2064, 2112), (2112, 2160), (2160, 2208), (2208, 2256), (2256, 2304), (2304, 2352), (2352, 2400), (2400, 2448), (2448, 2496), (2496, 2544), (2544, 2592), (2592, 2640), (2640, 2688), (2688, 2736), (2736, 2784), (2784, 2832), (2832, 2880), (2880, 2928), (2928, 2976), (2976, 3024), (3024, 3072), (3072, 3120), (3120, 3168), (3168, 3216), (3216, 3264), (3264, 3312), (3312, 3360), (3360, 3408), (3408, 3456), (3456, 3504), (3504, 3552), (3552, 360

In [ ]:
results.compute()